# CMPE 258 Team 7: Check-In 2 reproducible run (Colab T4)

**Robust Neural Image Watermarking with Failure-Guided Attack Sampling (FGAS)**

This notebook is the team's canonical environment. Teammates use different machines, so every reported number comes from this notebook on a **Colab T4 GPU** with the fixed seed in `configs/baseline.yaml`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

What it does:
1. Clones the repo and installs dependencies. Colab's preinstalled PyTorch is used.
2. Downloads DIV2K HR and caches the zips on Google Drive. Then it prepares the 600-train / 100-val subset.
3. Runs the unit tests.
4. Trains the uniform-sampling baseline (Experiments A and B).
5. Runs the standalone attack-wise evaluation, which gives Table 1 data.
6. Computes the FGAS probability update from real validation results (Experiment C, Table 2 data).
7. Runs a short FGAS training run. It shows the trainer consuming the new probabilities.
8. Copies everything to `results/` and to the team Google Drive folder.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Team Drive folder: checkpoints, run logs and the DIV2K zip cache go here.
# Change DRIVE_DIR if the shared team folder lives somewhere else in your Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CMPE258_Team7'
import os
os.makedirs(f'{DRIVE_DIR}/cache', exist_ok=True)

Mounted at /content/drive


In [3]:
REPO_URL = 'https://github.com/TaehunLeee5/CMPE258_Team7.git'
BRANCH = 'main'
%cd /content
!test -d CMPE258_Team7 || git clone -b {BRANCH} {REPO_URL}
%cd /content/CMPE258_Team7
!git pull --ff-only && git log -1 --oneline
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"

/content
Cloning into 'CMPE258_Team7'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 85 (delta 29), reused 85 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 40.90 KiB | 1.02 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/CMPE258_Team7
Already up to date.
3a6fd3d (HEAD -> main, origin/main, origin/HEAD) Add Colab T4 notebook as the reproducible team entry point and README
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
                                                                                              ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/cuda/__init__.py", line 653, in get_device_name
    return get_device_properties(device).name
           ~~~~~~

## 1. Data: DIV2K

In [ ]:
!python scripts/download_div2k.py --config configs/baseline.yaml --cache-dir {DRIVE_DIR}/cache
!python scripts/prepare_div2k.py --config configs/baseline.yaml

Extracting /content/drive/MyDrive/CMPE258_Team7/cache/DIV2K_train_HR.zip -> data/DIV2K
train: 800 images in data/DIV2K/DIV2K_train_HR
Extracting /content/drive/MyDrive/CMPE258_Team7/cache/DIV2K_valid_HR.zip -> data/DIV2K
valid: 100 images in data/DIV2K/DIV2K_valid_HR
train:  97% 581/600 [03:27<00:07,  2.39it/s]

In [ ]:
!python -m pytest -q

## 2. Uniform-sampling baseline (Experiments A and B)
All four attacks (noise, blur, brightness, resize) are sampled with probability 0.25. Validation runs after every epoch, under every attack separately. Expected time on a T4 is roughly 20-30 minutes.

In [ ]:
!python scripts/train.py --config configs/baseline.yaml

In [ ]:
# Standalone evaluation from the saved checkpoint: clean + every attack, PSNR/SSIM (Table 1 data)
!python scripts/evaluate.py --checkpoint runs/baseline_uniform/best.pt --out results/baseline_eval.json

## 3. FGAS probability update from the baseline (Experiment C)
failure = 1 - bit accuracy. p = min_prob + (1 - K * min_prob) * failure / sum(failure), with min_prob = 0.1.

In [ ]:
!python scripts/fgas_update.py --eval-json results/baseline_eval.json --config configs/fgas.yaml --out results/fgas_update.json

## 4. FGAS integration check
A short run that starts from the baseline weights. It applies an FGAS update first, then trains with the new distribution. Compare `prob_*` and `count_*` in `metrics.csv` with the uniform baseline.

In [ ]:
!python scripts/train.py --config configs/fgas.yaml --set train.epochs=2 experiment.name=fgas_integration_check

## 5. Collect results and copy to Drive

In [ ]:
import shutil, json, pathlib
out = pathlib.Path('results')
for run, files in {
    'baseline_uniform': ['metrics.csv', 'config.yaml', 'run_info.json', 'best_eval.json', 'best.pt'],
    'fgas_integration_check': ['metrics.csv', 'config.yaml', 'run_info.json', 'fgas_history.json'],
}.items():
    (out / run).mkdir(parents=True, exist_ok=True)
    for name in files:
        shutil.copy(pathlib.Path('runs') / run / name, out / run / name)

stamp = json.load(open('results/baseline_uniform/run_info.json'))['started_at'].replace(' ', '_').replace(':', '')
dest = f'{DRIVE_DIR}/colab_runs/{stamp}'
shutil.copytree('results', f'{dest}/results', dirs_exist_ok=True)
shutil.copytree('runs', f'{dest}/runs', dirs_exist_ok=True)
print('Copied results/ and runs/ to', dest)
!ls -R results

In [ ]:
ev = json.load(open('results/baseline_eval.json'))
print(f"Baseline (epoch {ev['epoch']}): PSNR {ev['image_quality']['psnr']:.2f} dB, SSIM {ev['image_quality']['ssim']:.4f}")
for a, v in ev['attacks'].items():
    print(f"  {a:<11} bit_acc {v['bit_acc']:.4f}  msg_acc {v['msg_acc']:.4f}")
print(open('results/fgas_update.csv').read())

## Optional: Experiment D, FGAS vs uniform continuation (Taehun)
Both runs start from the same baseline weights and train for the same number of epochs. Only the attack sampling differs. Uncomment both lines to run them (about 10 minutes each on a T4).

In [ ]:
# !python scripts/train.py --config configs/baseline_continue.yaml
# !python scripts/train.py --config configs/fgas.yaml
# !python scripts/evaluate.py --checkpoint runs/baseline_continue/best.pt --out results/baseline_continue_eval.json
# !python scripts/evaluate.py --checkpoint runs/fgas_continue/best.pt --out results/fgas_continue_eval.json